# Jupyter Notebook Tiền xử lý & EDA (THỊNH)

- **Người thực hiện:** THỊNH
- **Danh mục:** Nhà Cửa & Đời Sống
- **Công cụ:** Pandas, NumPy, Plotly Express

**Nhiệm vụ:**
1. Đọc dữ liệu thô (Raw).
2. Làm sạch & Ép kiểu dữ liệu.
3. Tính Feature Engineering (Các cột derived mới).
4. **Mục tiêu 1:** Trực quan hóa Chất lượng trang -> Doanh số Mall.
5. **Mục tiêu 2:** Trực quan hóa Loại hình giao hàng.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

pd.set_option('display.max_columns', None)

df_fact = pd.read_csv('../data/raw/fact_product_tiki_20260326_thinh.csv', encoding='utf-8-sig')
df_shop = pd.read_csv('../data/raw/dim_shop_tiki_20260326_thinh.csv', encoding='utf-8-sig')

df = pd.merge(df_fact, df_shop, on=['shop_id', 'platform_id'], how='left')
print(f"[PRE-CLEAN] Dữ liệu ban đầu: {df.shape[0]} hàng, {df.shape[1]} cột")
df.head(2)

In [ ]:
numeric_cols = ['price_current', 'price_original', 'discount_percent', 'rating']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

int_cols = ['sold_count', 'stock', 'review_count', 'image_count', 'review_with_image_count', 'five_star_with_image_count']
for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('int64')

bool_cols = ['price_ends_with_9', 'has_video', 'is_freeship', 'has_freeship_xtra_label', 'has_coinback_label', 'has_voucher_label', 'is_mall']
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(bool)

df['price_original'] = df.apply(lambda row: row['price_current'] if row['price_original'] == 0 else row['price_original'], axis=1)
df = df[(df['price_current'] > 1000) & (df['price_current'] <= 5000000000)].copy()
print("[POST-CLEAN] Dữ liệu sạch:", df.shape)

In [ ]:
def assign_price_bucket(price):
    if price < 100000: return '<100k'
    if price < 500000: return '100k-500k'
    if price < 1000000: return '500k-1M'
    if price < 5000000: return '1M-5M'
    return '>5M'
df['price_bucket_der'] = df['price_current'].apply(assign_price_bucket)

df['content_quality_score'] = np.clip(df['image_count'], 0, 5) + (df['has_video'].astype(int) * 2) + (df['rating'] >= 4.5).astype(int) * 2

def parse_delivery(text):
    if pd.isna(text) or str(text).strip() == "": return "Tiêu chuẩn (Có thể thiếu)"
    text = str(text).lower()
    if '2h' in text or 'now' in text or 'pro' in text or 'hôm nay' in text:
        return "Giao Nhanh/Hỏa Tốc"
    return "Giao Tiêu Chuẩn"
df['delivery_type_der'] = df['estimated_delivery_days'].apply(parse_delivery)

In [ ]:
df_mall = df[df['is_mall'] == True].copy()
fig1 = px.box(
    df_mall, 
    x="content_quality_score", 
    y="sold_count", 
    points="all", 
    color="has_video",
    title="[THỊNH] - Ảnh Hưởng Của Điểm Trang Tới Doanh Số Khối Mall",
    log_y=True
)
fig1.show()

In [ ]:
grouped = df.groupby('delivery_type_der')[['sold_count', 'review_count']].mean().reset_index()
fig2 = px.bar(
    grouped.sort_values(by='sold_count', ascending=False), 
    x='delivery_type_der', 
    y='sold_count',
    text_auto=True,
    title="[THỊNH] - Lượt bán theo Loại Hình Giao Hàng",
    color='delivery_type_der'
)
fig2.update_traces(textposition='outside')
fig2.show()

In [ ]:
import os
out_dir = '../data/processed'
if not os.path.exists(out_dir): os.makedirs(out_dir)
out_file = f"{out_dir}/thinh_cleaned_data_preview.csv"
df.to_csv(out_file, index=False, encoding='utf-8-sig')
print("Đã lưu bản tiền xử lý vào:", out_file)